In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [3]:
!pip install ultralytics opencv-python-headless pandas numpy -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 20.4 MB/s eta 0:00:00a 0:00:01


In [4]:
#importing all libraries 
import os
import glob
import shutil
from ultralytics import YOLO
import cv2
import matplotlib.pyplot as plt
print(os.listdir("/kaggle/input"))

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
['datasets']


In [12]:
DATASET_ROOT = "/kaggle/input/datasets/garimatrip/cricket-dataset-v2" # change this

print(os.listdir(DATASET_ROOT))
print(os.listdir(f"{DATASET_ROOT}/train"))
print(os.listdir(f"{DATASET_ROOT}/valid"))

['README.dataset.txt', 'README.roboflow.txt', 'data.yaml', 'valid', 'test', 'train']
['labels', 'images']
['labels', 'images']


In [13]:


SRC = "/kaggle/input/datasets/garimatrip/cricket-dataset-v2"
DST = "/kaggle/working/cricket_dataset"

if os.path.exists(DST):
    shutil.rmtree(DST)

shutil.copytree(SRC, DST)
print("Copied to:", DST)
print(os.listdir(DST))

Copied to: /kaggle/working/cricket_dataset
['data.yaml', 'README.roboflow.txt', 'test', 'valid', 'README.dataset.txt', 'train']


In [14]:


sample = sorted(glob.glob(f"{DST}/train/labels/*.txt"))[0]
print("Sample label file:", sample)
print(open(sample).read())

Sample label file: /kaggle/working/cricket_dataset/train/labels/0_jpg.rf.ccafee537ea14690f32994fe72b6a649.txt
0 0.1 0.10703125 0.040625 0.02890625


In [15]:
#removing stump as i need to only detect balls 


def keep_only_ball(label_dir):
    files = glob.glob(os.path.join(label_dir, "*.txt"))

    before = 0
    after = 0

    for path in files:
        with open(path, "r") as f:
            lines = [line.strip() for line in f if line.strip()]

        before += len(lines)

        new_lines = []
        for line in lines:
            cls = int(line.split()[0])
            if cls == 0:   # keep only ball
                new_lines.append(line)

        after += len(new_lines)

        with open(path, "w") as f:
            if new_lines:
                f.write("\n".join(new_lines) + "\n")

    print(f"{label_dir}")
    print("Before:", before)
    print("After :", after)


keep_only_ball("/kaggle/working/cricket_dataset/train/labels")
keep_only_ball("/kaggle/working/cricket_dataset/valid/labels")

if os.path.exists("/kaggle/working/cricket_dataset/test/labels"):
    keep_only_ball("/kaggle/working/cricket_dataset/test/labels")

/kaggle/working/cricket_dataset/train/labels
Before: 13295
After : 8862
/kaggle/working/cricket_dataset/valid/labels
Before: 1561
After : 938
/kaggle/working/cricket_dataset/test/labels
Before: 754
After : 418


In [ ]:
#we can keep the stump images as they can act as negative samples #robust training 
label_files = glob.glob("/kaggle/working/cricket_dataset/train/labels/*.txt")

non_empty = 0
empty = 0

for f in label_files:
    content = open(f).read().strip()
    if content:
        non_empty += 1
    else:
        empty += 1

print("non-empty:", non_empty)
print("empty:", empty)


In [16]:
# so 80% is non empty can keep the stump images too

#checking if any stump do exist 

# label_files = glob.glob("/kaggle/working/cricket_dataset/train/labels/*.txt")
label_files = glob.glob("/kaggle/working/cricket_dataset/valid/labels/*.txt")
stump_found = False

for file in label_files:
    with open(file, "r") as f:
        for line in f:
            if line.strip():
                cls = int(line.split()[0])
                if cls == 1:
                    stump_found = True
                    print("Stump found in:", file)
                    break

if not stump_found:
    print("No stump found")


No stump found


In [17]:
data_yaml = """
path: /kaggle/working/cricket_dataset

train: train/images
val: valid/images
test: test/images

nc: 1
names: ['ball']
"""

with open("/kaggle/working/data.yaml", "w") as f:
    f.write(data_yaml)

print(open("/kaggle/working/data.yaml").read())


path: /kaggle/working/cricket_dataset

train: train/images
val: valid/images
test: test/images

nc: 1
names: ['ball']



In [18]:
base = "/kaggle/working/cricket_dataset"
print("train images:", os.path.exists(f"{base}/train/images"))
print("train labels:", os.path.exists(f"{base}/train/labels"))
print("valid images:", os.path.exists(f"{base}/valid/images"))
print("valid labels:", os.path.exists(f"{base}/valid/labels"))
print("test images:", os.path.exists(f"{base}/test/images"))
print("data.yaml exists:", os.path.exists("/kaggle/working/data.yaml"))

train images: True
train labels: True
valid images: True
valid labels: True
test images: True
data.yaml exists: True


In [19]:
model = YOLO("yolov8s.pt")

In [20]:
results = model.train(
    data="/kaggle/working/data.yaml",

    epochs=120,
    imgsz=960,
    batch=8,
    device=0,
    optimizer="AdamW",

    lr0=0.001,
    lrf=0.1,
    warmup_epochs=3,
    weight_decay=0.0005,

    patience=20,

    mosaic=0.5,
    mixup=0.0,
    scale=0.5,
    translate=0.1,
    fliplr=0.5,
    flipud=0.0,

    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=2.0,

    project="/kaggle/working/runs",
    name="cricket_ball",
    save=True,
    plots=True,
    val=True,
)

Ultralytics 8.4.32 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/data.yaml, degrees=2.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=120, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=960, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.1, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=0.5, multi_scale=0.0, name=cricket_ball, nbs=64, nms=False, opset=None, optimize=False, optimizer=AdamW, overlap_mask=True, patience=20, perspective

KeyboardInterrupt: 

In [21]:
import shutil

shutil.make_archive('/kaggle/working/results_NEW_zip', 'zip', '/kaggle/working/runs')

'/kaggle/working/results_NEW_zip.zip'